In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
train_data = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv")
test_data = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv")

oil=pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv")
holidays = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv")
stores = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv")
transactions = pd.read_csv("/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv")

print("Train data Shape",train_data.shape)
print(train_data.head())

Train data Shape (3000888, 6)
   id        date  store_nbr      family  sales  onpromotion
0   0  2013-01-01          1  AUTOMOTIVE    0.0            0
1   1  2013-01-01          1   BABY CARE    0.0            0
2   2  2013-01-01          1      BEAUTY    0.0            0
3   3  2013-01-01          1   BEVERAGES    0.0            0
4   4  2013-01-01          1       BOOKS    0.0            0


In [3]:
train_data["date"]=pd.to_datetime(train_data["date"])
test_data["date"]=pd.to_datetime(test_data["date"])

In [4]:
print(train_data.isnull().sum())
print(oil.isnull().sum())

id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64
date           0
dcoilwtico    43
dtype: int64


In [5]:
oil["dcoilwtico"]= oil["dcoilwtico"].fillna(method='ffill')
oil["dcoilwtico"]= oil["dcoilwtico"].fillna(method='bfill')

print(oil.isnull().sum())

date          0
dcoilwtico    0
dtype: int64


/tmp/ipykernel_24/3377208407.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  oil["dcoilwtico"]= oil["dcoilwtico"].fillna(method='ffill')
/tmp/ipykernel_24/3377208407.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  oil["dcoilwtico"]= oil["dcoilwtico"].fillna(method='bfill')


In [6]:
# convert the date column of all files into datetime
train_data['date'] = pd.to_datetime(train_data['date'])
test_data['date'] = pd.to_datetime(test_data['date'])
oil['date'] = pd.to_datetime(oil['date'])
holidays['date'] = pd.to_datetime(holidays['date'])

# Ab merge karein (Ab error nahi aayega kyunke sabka type same hai)

# 1. Stores merge
train = train_data.merge(stores, on='store_nbr', how='left')

# 2. Oil merge
train = train.merge(oil, on='date', how='left')

# 3. Holidays merge 
holidays_unique = holidays.drop_duplicates(subset=['date'])
train = train.merge(holidays_unique, on='date', how='left')


test = test_data.merge(stores, on='store_nbr', how='left')
test = test.merge(oil, on='date', how='left')
test = test.merge(holidays_unique, on='date', how='left')

print("Success! Final columns:", train.columns)

Success! Final columns: Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city',
       'state', 'type_x', 'cluster', 'dcoilwtico', 'type_y', 'locale',
       'locale_name', 'description', 'transferred'],
      dtype='object')


In [7]:
train=train.rename(columns={'type_x':'store_type','type_y':'holiday_type'})
test=test.rename(columns={'type_x':'store_type','type_y':'holiday_type'})

In [8]:
def extract_date_features(df):
    df['day_of_week']=df['date'].dt.dayofweek
    df['month']=df['date'].dt.month
    df['year']=df['date'].dt.year
    df['is_weekend']=df['day_of_week'].isin([5,6]).astype(int)
    return df

train = extract_date_features(train)
test = extract_date_features(test)

In [9]:
train['holiday_type']=train['holiday_type'].fillna('Work_day')
train['locale'] = train['locale'].fillna('National')

train['dcoilwtico']=train['dcoilwtico'].fillna(method='ffill').fillna(method='bfill')
test['dcoilwtico']=test['dcoilwtico'].fillna(method='ffill').fillna(method='bfill')


/tmp/ipykernel_24/2209823869.py:4: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  train['dcoilwtico']=train['dcoilwtico'].fillna(method='ffill').fillna(method='bfill')
/tmp/ipykernel_24/2209823869.py:5: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  test['dcoilwtico']=test['dcoilwtico'].fillna(method='ffill').fillna(method='bfill')


In [10]:
from sklearn.preprocessing import LabelEncoder

# list of Text columns
categorical_cols = ['family', 'city', 'state', 'store_type', 'holiday_type', 'locale', 'locale_name', 'transferred']

le = LabelEncoder()

for col in categorical_cols:
   #First, ensure that the column is a string and not NaN.

    train[col] = train[col].astype(str).fillna('Unknown')
    test[col] = test[col].astype(str).fillna('Unknown')
    
    # Fit and Transform
    train[col] = le.fit_transform(train[col])
    
   
    test[col] = test[col].map(lambda s: le.transform([s])[0] if s in le.classes_ else -1)

print("Encoding Done!")
print(train[categorical_cols].dtypes) 

Encoding Done!
family          int64
city            int64
state           int64
store_type      int64
holiday_type    int64
locale          int64
locale_name     int64
transferred     int64
dtype: object


In [11]:
#Keep only the numeric columns.
#Drop the Date and Description columns because they are objects.
drop_cols = ['date', 'description']

train_final = train.drop(drop_cols, axis=1)
test_final = test.drop(drop_cols, axis=1)

# Ensure karein ke 'id' train mein na ho lekin test ki 'id' humein submission ke liye chahiye
if 'id' in train_final.columns:
    train_final = train_final.drop(['id'], axis=1)

In [12]:
drop_cols = ['date', 'description', 'id']
X = train.drop(drop_cols + ['sales'], axis=1)
y = train['sales']
X_test_final = test.drop(drop_cols, axis=1, errors='ignore')

In [13]:
import xgboost as xgb
from sklearn.model_selection import train_test_split

# separating the  X (Features) aur y (Target - Sales)
X = train_final.drop(['sales'], axis=1)
y = train_final['sales']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# creating Model 
model = xgb.XGBRegressor(
    n_estimators=500, 
    learning_rate=0.1, 
    max_depth=6, 
    tree_method='hist', 
    device="cuda"      
)

print("Training has started...")
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)
print("Model Trained successfuly!")

Training has started...
[0]	validation_0-rmse:1047.69908
[100]	validation_0-rmse:412.14208
[200]	validation_0-rmse:370.73827
[300]	validation_0-rmse:353.70474
[400]	validation_0-rmse:341.81777
[499]	validation_0-rmse:332.96068
Model Trained successfuly!


In [14]:
preds= model.predict(X_test_final)
preds = np.maximum(0,preds)

submission = pd.DataFrame({
    'id':test_data['id'],
    'sales':preds
})

submission.to_csv('submission_csv',index=False)
print("Submission file 'submission.csv' is ready!")

Submission file 'submission.csv' is ready!


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [16:50:09] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
